# 06 - 本地 RQAlphaPlus 策略回测
本 Notebook **只在本地授权环境运行**，不在 Colab 运行。它从 YAML 的 `backtest` 段读取全部参数并显式传给回测程序，命令行参数仅用于有意覆盖配置。

In [ ]:
from pathlib import Path
import os

if not Path('configs/daily/training.yaml').exists():
    repo = os.environ.get('ALPHAMINING_REPO_URL', 'https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

## 1. 导入 Colab 训练产物

In [ ]:
from zipfile import ZipFile

archive_path = Path('alphamining_colab_outputs.zip')
prediction_path = Path('results/lightgbm/prediction_score.csv')
if not prediction_path.exists():
    if not archive_path.exists():
        raise FileNotFoundError('请把 Colab 下载的 alphamining_colab_outputs.zip 放到仓库根目录')
    with ZipFile(archive_path) as archive:
        archive.extractall('.')
assert prediction_path.exists(), '压缩包中缺少 prediction_score.csv'
print('已加载 Colab 预测分数:', prediction_path.resolve())

## 2. 加载并确认回测配置
默认读取本地正式配置。若需要复现 Colab 冻结配置，可把路径改为 `results/colab_training_config.yaml`。

In [ ]:
import json
from rqalpha_strategy.run_backtest import load_backtest_settings

BACKTEST_CONFIG_PATH = Path('configs/daily/training.yaml')
backtest_settings = load_backtest_settings(BACKTEST_CONFIG_PATH)
print('回测配置来源:', BACKTEST_CONFIG_PATH.resolve())
print(json.dumps(backtest_settings, ensure_ascii=False, indent=2))

## 3. 运行 RQAlphaPlus
执行前会打印最终生效参数，并保存到 `results/backtest_report/backtest_effective_config.json`。

In [ ]:
import subprocess
import sys

RQALPHA_PLUS_BUNDLE = Path('~/.rqalpha-plus/bundle').expanduser()  # 修改为本地已授权的数据包路径
command = [
    sys.executable, '-m', 'rqalpha_strategy.run_backtest',
    '--config', str(BACKTEST_CONFIG_PATH),
    '--bundle', str(RQALPHA_PLUS_BUNDLE),
    '--predictions', str(prediction_path),
    '--output-dir', 'results/backtest_report',
]
print('执行命令:', ' '.join(command))
subprocess.run(command, check=True)

## 4. 校验实际生效参数并展示净值曲线

In [ ]:
effective_path = Path('results/backtest_report/backtest_effective_config.json')
effective = json.loads(effective_path.read_text(encoding='utf-8'))
for key in backtest_settings:
    assert effective[key] == backtest_settings[key], f'{key} 未按配置生效: {effective[key]} != {backtest_settings[key]}'
print('参数校验通过:', json.dumps(effective, ensure_ascii=False, indent=2))

from IPython.display import Image, display
display(Image('results/backtest_report/equity_curve.png'))